# BDC Satria Data 2026 — Klasifikasi Citra Sampah
### ViT-H/14 SWAG — Eksplorasi Model (belum predict test)

**Strategi split (tanpa holdout, sesuai kesepakatan):**
- `StratifiedShuffleSplit(n_splits=3, test_size=0.10)` → tiap fold rasio train:val = 90:10
- Metrik utama: **Macro F1-Score**
- Class weighting (`balanced`) dihitung ulang tiap fold → tangani imbalance (Organic 47% / Recyclable 38% / Electronic 15%)

**Tracking misclassified → `misclassified_report_vith14.xlsx` (2 sheet):** Detail + Summary (kolom sama persis dgn notebook ConvNeXt kamu).

**Catatan model:** ViT-H/14 SWAG tersedia langsung di `torchvision` (tidak perlu timm). Bobot `IMAGENET1K_SWAG_LINEAR_V1` pakai resolusi **224** (cocok dengan pipeline kamu). Backbone di-freeze, hanya classifier head yang dilatih.

**Ukuran:** ViT-H/14 (~632M params) jauh lebih besar dari ViT-L/16 (~305M) dan ViT-B/16 (~86M) — attention makin boros memory & compute (patch 14 → lebih banyak token per gambar). Bobot pretrained ~2.4GB — download awal bisa lebih lama, dan batch size perlu diturunkan dibanding ViT-L/16.

## 1. Setup & Konfigurasi

In [1]:
from __future__ import annotations
import os
import json
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms, models
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

In [2]:
# Fix Windows SSL bug (ASN1: NOT_ENOUGH_DATA) saat download pretrained weights
import ssl
import certifi

ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())


In [ ]:
# === PATH DATASET ===
DATA_ROOT = Path(r"C:\Users\MyPC PRO\Downloads\BDC2026")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"          # belum dipakai di notebook ini
MODELS_DIR = DATA_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Folder kelas: prefix angka menentukan urutan label (0=Recyclable, 1=Electronic, 2=Organic)
CLASS_FOLDERS = ["0_Recyclable", "1_Electronic", "2_Organic"]
CLASS_NAMES = ["Recyclable", "Electronic", "Organic"]
NUM_CLASSES = len(CLASS_NAMES)

# === CONFIG TRAINING ===
IMG_SIZE = 224
BATCH_SIZE = 30          # ViT-H/14 (~632M) jauh lebih berat dari ViT-L/16 - turun dari 43 utk jaga2 VRAM 12GB, naikkan kalau masih longgar
NUM_EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 4             # early stopping berdasarkan macro F1 val
N_FOLDS = 3
FOLD_VAL_FRAC = 0.10     # 90/10 tiap fold
SEED = 42
NUM_WORKERS = 8          # paralelkan image decode/resize (32 core CPU tersedia); Dataset di dataset_utils.py biar worker Windows gak hang

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Model: ViT-H/14 SWAG (IMAGENET1K_SWAG_LINEAR_V1, 224)")

Device: cuda
GPU: NVIDIA GeForce RTX 3060
VRAM: 12.9 GB
Model: ViT-H/14 SWAG (IMAGENET1K_SWAG_LINEAR_V1, 224)


## 2. Index Dataset dari Folder

In [4]:
def index_dataset(train_dir: Path, class_folders: list[str]):
    records = []
    for label_idx, folder_name in enumerate(class_folders):
        folder = train_dir / folder_name
        if not folder.is_dir():
            raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        files = [p for p in folder.iterdir() if p.suffix.lower() in exts]
        for p in files:
            records.append({"path": str(p), "label": label_idx, "class_name": CLASS_NAMES[label_idx]})
    return pd.DataFrame(records)

full_df = index_dataset(TRAIN_DIR, CLASS_FOLDERS)
print(f"Total citra ter-index: {len(full_df)}")
print(full_df["class_name"].value_counts())

Total citra ter-index: 26527
class_name
Organic       12567
Recyclable     9999
Electronic     3961
Name: count, dtype: int64


## 3. Dataset & Transforms

In [5]:
# TrashDataset & get_transforms dipindah ke dataset_utils.py (wajib jadi modul .py asli
# biar DataLoader num_workers>0 gak hang di Windows/Jupyter - spawn worker perlu import modul nyata)
from dataset_utils import TrashDataset, get_transforms

## 4. Model — ViT-H/14 SWAG (backbone freeze, head-only training)
Struktur head sama seperti ViT-B/16 kamu: `model.heads.head` diganti dengan Dropout + Linear.

In [6]:
def build_model(num_classes: int):
    print("  Loading ViT-H/14 SWAG weights (download ~2.4GB pertama kali)...")
    model = models.vit_h_14(weights=models.ViT_H_14_Weights.IMAGENET1K_SWAG_LINEAR_V1)

    # Freeze backbone (patch embed + encoder)
    for p in model.conv_proj.parameters():
        p.requires_grad = False
    for p in model.encoder.parameters():
        p.requires_grad = False

    # Ganti classifier head
    in_features = model.heads.head.in_features
    model.heads = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features=in_features, out_features=num_classes, bias=True),
    )
    return model.to(DEVICE)


_tmp = build_model(NUM_CLASSES)
n_total = sum(p.numel() for p in _tmp.parameters())
n_train = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"Total params: {n_total:,}")
print(f"Trainable params (head): {n_train:,}")
del _tmp
torch.cuda.empty_cache()

  Loading ViT-H/14 SWAG weights (download ~2.4GB pertama kali)...
Total params: 630,768,643
Trainable params (head): 5,123


## 5. Train & Evaluate

In [7]:
scaler = GradScaler("cuda", enabled=(DEVICE == "cuda"))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_paths, all_probs = [], [], [], []
    for imgs, labels, paths in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        running_loss += loss.item()

        probs = F.softmax(outputs.float(), dim=1)
        preds = probs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_paths.extend(paths)
        all_probs.extend(probs.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1, np.array(all_labels), np.array(all_preds), all_paths, np.array(all_probs)

## 6. Cross Validation 3-Fold + Kumpulkan Misclassified

In [ ]:
sss = StratifiedShuffleSplit(n_splits=N_FOLDS, test_size=FOLD_VAL_FRAC, random_state=SEED)

fold_results = []
detail_records = []
eval_counter = defaultdict(int)

for fold, (train_idx, val_idx) in enumerate(sss.split(full_df, full_df["label"]), start=1):
    print(f"\n{'='*64}\n---> FOLD {fold}/{N_FOLDS} <---\n{'='*64}")
    train_df = full_df.iloc[train_idx]
    val_df = full_df.iloc[val_idx]

    weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["label"].values)
    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print(f"  Class weights: {dict(zip(CLASS_NAMES, weights.round(3)))}")

    train_loader = DataLoader(TrashDataset(train_df, IMG_SIZE, get_transforms(IMG_SIZE, True)),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=(NUM_WORKERS > 0), prefetch_factor=4 if NUM_WORKERS > 0 else None)
    val_loader = DataLoader(TrashDataset(val_df, IMG_SIZE, get_transforms(IMG_SIZE, False)),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                            persistent_workers=(NUM_WORKERS > 0), prefetch_factor=4 if NUM_WORKERS > 0 else None)

    model = build_model(NUM_CLASSES)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                  lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_f1 = 0.0
    epochs_no_improve = 0
    best_path = MODELS_DIR / f"vith14_swag_fold{fold}.pth"

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_f1, _, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_f1)

        marker = ""
        if val_f1 > best_f1:
            best_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            marker = " [*] best"
        else:
            epochs_no_improve += 1

        print(f"  Epoch {epoch:2d}/{NUM_EPOCHS} | train_loss {tr_loss:.4f} F1 {tr_f1:.4f} | "
              f"val_loss {val_loss:.4f} F1 {val_f1:.4f}{marker}")

        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping (tidak ada peningkatan {PATIENCE} epoch).")
            break

    model.load_state_dict(torch.load(best_path, weights_only=True))
    _, final_f1, y_true, y_pred, paths, probs = evaluate(model, val_loader, criterion)
    print(f"  >> Fold {fold} best macro F1: {best_f1:.4f} | re-eval F1: {final_f1:.4f}")

    fold_results.append({"fold": fold, "best_macro_f1": best_f1})

    for i, path in enumerate(paths):
        eval_counter[path] += 1
        if y_true[i] != y_pred[i]:
            p = probs[i]
            detail_records.append({
                "fold": fold,
                "filepath": path,
                "true_label": CLASS_NAMES[y_true[i]],
                "predicted_label": CLASS_NAMES[y_pred[i]],
                "confidence_pct": round(float(p[y_pred[i]]) * 100, 2),
                "true_label_confidence_pct": round(float(p[y_true[i]]) * 100, 2),
                "prob_recyclable_pct": round(float(p[0]) * 100, 2),
                "prob_electronic_pct": round(float(p[1]) * 100, 2),
                "prob_organic_pct": round(float(p[2]) * 100, 2),
            })

    del model, train_loader, val_loader
    torch.cuda.empty_cache()


---> FOLD 1/3 <---
  Class weights: {'Recyclable': np.float64(0.884), 'Electronic': np.float64(2.232), 'Organic': np.float64(0.704)}
  Loading ViT-H/14 SWAG weights (download ~2.4GB pertama kali)...


## 7. Ringkasan Hasil 3-Fold

In [ ]:
f1s = [r["best_macro_f1"] for r in fold_results]
print("Ringkasan 3-Fold CV (macro F1):")
for r in fold_results:
    print(f"  Fold {r['fold']}: {r['best_macro_f1']:.4f}")
print(f"\nRata-rata macro F1: {np.mean(f1s):.4f} (+/- {np.std(f1s):.4f})")
print(f"\nPembanding — ViT-B/16 SWAG kamu: ~0.9636")

## 8. Bangun Laporan Misclassified (`.xlsx`, 2 sheet)

In [ ]:
detail_df = pd.DataFrame(detail_records, columns=[
    "fold", "filepath", "true_label", "predicted_label",
    "confidence_pct", "true_label_confidence_pct",
    "prob_recyclable_pct", "prob_electronic_pct", "prob_organic_pct",
])

summary_rows = []
if len(detail_df) > 0:
    for path, g in detail_df.groupby("filepath"):
        folds_wrong = sorted(g["fold"].unique().tolist())
        summary_rows.append({
            "filepath": path,
            "true_label": g["true_label"].iloc[0],
            "folds_wrong": ",".join(map(str, folds_wrong)),
            "total_wrong": len(g),
            "total_evaluated": eval_counter[path],
            "error_rate": round(len(g) / eval_counter[path], 3) if eval_counter[path] else None,
            "avg_confidence_pct": round(g["confidence_pct"].mean(), 2),
            "avg_true_label_confidence_pct": round(g["true_label_confidence_pct"].mean(), 2),
            "avg_prob_recyclable_pct": round(g["prob_recyclable_pct"].mean(), 2),
            "avg_prob_electronic_pct": round(g["prob_electronic_pct"].mean(), 2),
            "avg_prob_organic_pct": round(g["prob_organic_pct"].mean(), 2),
        })

summary_df = pd.DataFrame(summary_rows, columns=[
    "filepath", "true_label", "folds_wrong", "total_wrong", "total_evaluated", "error_rate",
    "avg_confidence_pct", "avg_true_label_confidence_pct",
    "avg_prob_recyclable_pct", "avg_prob_electronic_pct", "avg_prob_organic_pct",
])
if len(summary_df) > 0:
    summary_df = summary_df.sort_values(["error_rate", "total_wrong"], ascending=False).reset_index(drop=True)

report_path = OUTPUT_DIR / "misclassified_report_vith14.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    detail_df.to_excel(writer, sheet_name="Detail", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"[SAVED] {report_path}")
print(f"  Detail  : {len(detail_df)} baris")
print(f"  Summary : {len(summary_df)} gambar unik")
if len(summary_df) > 0:
    print("\n5 gambar paling problematik:")
    print(summary_df.head(5).to_string(index=False))

## Langkah selanjutnya
1. Bandingkan rata-rata macro F1 ViT-H/14 di atas dengan ViT-L/16, ViT-B/16 (~0.9636) & ConvNeXt V2-Large.
2. Kalau lanjut stacking: ganti `StratifiedShuffleSplit` → `StratifiedKFold` supaya OOF prediction bersih.